# 예제 franka_ex07: FR3 Pick & Place

Franka FR3 의 7-DOF 팔과 Franka Hand 그리퍼를 통합해서 가상 물체를 한 위치에서
잡아 다른 위치로 옮기는 시퀀스.

## 이 노트북이 강조하는 것 — `plan` → FK 미리보기 → `execute` 워크플로

ex05 에서 본 "계획 ≠ 실행" 분리 모델은 cartesian 직선 보간 (`compute_cartesian_path`) 한정이었다.
ex07 은 그 분리 모델을 **일반 MoveGroup pose / joint goal** 에까지 확장하고, 사이에
**FK 미리보기** 단계를 끼워 넣는다.

| 단계 | 도구 | 결과 |
|---|---|---|
| **계획만** | `MoveGroup` 액션 + `PlanningOptions(plan_only=True)` | trajectory 객체 |
| **미리보기** | `compute_fk` 서비스로 trajectory 의 각 시점을 EE 좌표로 환산 → `LINE_STRIP` 발행 | RViz 에 곡선 경로 |
| **실행** | `ExecuteTrajectory` 액션 (ex05 와 동일) | 실제 컨트롤러로 전송 |

ex05 의 cartesian 보간은 직선이 보장되어 미리보기가 단순했지만, MoveGroup 의 PRM/RRT
경로는 **직선이 아닌 곡선** 이라 끝단이 어디로 끌려갈지 예측이 어렵다 — 그래서
실행 전에 FK 로 곡선 모양을 먼저 그려보고 확인하는 게 유용하다.

이 셋이 `plan_viz_execute()` 한 함수로 묶이고, 9 단계 Pick & Place 시퀀스가 모두 이 함수
한 줄 호출로 진행된다.

## 이전 예제와의 관계
- ex03 / ex04: `MoveGroup` 액션 — plan + execute 통합 (이 예제는 분리)
- ex05: cartesian 한정 분리 모델 + fraction 게이트 (이 예제는 일반 pose / joint goal)
- ex06: 그리퍼 단독 (`FollowJointTrajectory`) — 여기선 팔과 통합

## 노트북 구성
1. **로봇 / 그리퍼 상수**
2. **핵심 — plan → 미리보기 → execute** ← 이 노트북의 본질
3. **핵심을 쓰기 위한 설정** — ROS init, 노드, 클라이언트 (FK 포함), SRDF, Pose / MoveGroup / 그리퍼 / 마커 헬퍼
4. **Pick & Place 시나리오** — 9 단계

## 실행 절차

이 노트북은 별도로 띄운 MoveIt + RViz 의 `move_group` 액션 서버에 클라이언트로 붙는다.

> ⚠ 다른 로봇용 MoveIt launch 가 떠 있으면 같은 토픽으로 충돌할 수 있다.
> 시작 전에 `pgrep -af 'ros2 launch'` 로 잔존 프로세스가 없는지 확인하자.

### 터미널 1 — Franka FR3 (Gazebo Sim) + MoveIt + RViz 기동

```bash
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
ros2 launch franka_tutorials franka_gazebo_moveit.launch.py
```

RViz 가 뜨면 **`MarkerArray` Display 를 추가하고 Topic 을 `/pick_place_markers` 로 설정**한다.
Fixed Frame 은 `fr3_link0` 로 둔다. Planning Scene Display 도 추가하면 충돌 객체가 같이 보인다.

### 터미널 2 — Jupyter 기동

```bash
source ~/venv/ros_jazzy/bin/activate
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
cd ~/robot_arm/src/robotarm_tutorials/robot_arm_tutorials/robot_arm_tutorials
jupyter lab franka_ex07_pick_and_place.ipynb
```

셀을 위에서 아래로 순서대로 실행한다 (`Shift+Enter`).

## 1. 로봇 / 그리퍼 상수

핵심 함수와 설정 양쪽이 모두 참조하므로 가장 먼저 정의한다.

In [1]:
PLANNING_GROUP    = 'fr3_arm'
REFERENCE_FRAME   = 'fr3_link0'
END_EFFECTOR_LINK = 'fr3_hand_tcp'
ARM_JOINTS        = ['fr3_joint1', 'fr3_joint2', 'fr3_joint3',
                     'fr3_joint4', 'fr3_joint5', 'fr3_joint6',
                     'fr3_joint7']

# FR3 gripper: fr3_gripper_controller (JointTrajectoryController)
# 액티브 조인트는 fr3_finger_joint1 (prismatic, 0.0=닫힘, 0.04=한쪽 손가락 최대 열림).
# fr3_finger_joint2 는 mimic 이라 컨트롤러에는 안 들어감.
GRIPPER_JOINT  = 'fr3_finger_joint1'
GRIPPER_ACTION = '/fr3_gripper_controller/follow_joint_trajectory'
GRIPPER_OPEN   = 0.04
GRIPPER_CLOSED = 0.0

MARKER_TOPIC = '/pick_place_markers'

## 2. 핵심 — `plan` → FK 미리보기 → `execute`

이 노트북에서 가장 먼저 정의해야 하는 함수들. 다섯이 한 묶음으로 본 예제의 본질을 이룬다.

- `plan_to_pose_goal()` / `plan_to_joint_goal()` — `PlanningOptions(plan_only=True)` 로 trajectory 만 받기
- `trajectory_to_ee_path()` — `compute_fk` 서비스로 EE 좌표 경로 환산
- `publish_ee_path()` — `LINE_STRIP` 으로 RViz 미리보기
- `execute_trajectory()` — 계획된 trajectory 만 실행
- `plan_viz_execute()` — 위 넷을 한 줄로 묶은 통합 함수

이 함수들은 setup 셀에서 만드는 `node` / `move_client` / `execute_client` / `fk_client` /
`marker_pub` / `_markers` / `make_*_constraint` / `make_plan_request` 등을 참조한다.
함수 *정의* 시점엔 lookup 하지 않으므로 객체가 아직 없어도 OK — 호출은 4 절(시나리오)에서
일어난다.

### 2-1. 핵심에 필요한 import

In [2]:
import rclpy
from moveit_msgs.action import MoveGroup, ExecuteTrajectory
from moveit_msgs.srv import GetPositionFK
from moveit_msgs.msg import (
    Constraints, MoveItErrorCodes,
    MotionPlanRequest, PlanningOptions, RobotState,
)
from visualization_msgs.msg import Marker
from geometry_msgs.msg import Point

### 2-2. `send_move_goal()` + `plan_to_*_goal()` — 계획만 받기

`MoveGroup` 액션의 `PlanningOptions(plan_only=True)` 가 핵심.
`plan_only=True` 면 액션 서버는 trajectory 만 만들어서 돌려주고 컨트롤러로 보내지 않는다.

`plan_to_*_goal()` 은 joint / pose 목표용 wrapper — 받은 trajectory 를 그대로 반환.

In [3]:
def send_move_goal(req: MotionPlanRequest, plan_only: bool = False):
    '''MoveGroup 액션 호출. plan_only=True 면 계획만, False 면 plan+execute 까지.'''
    goal = MoveGroup.Goal()
    goal.request = req
    goal.planning_options = PlanningOptions(
        plan_only=plan_only,
        replan=not plan_only,
        replan_attempts=3 if not plan_only else 0,
    )
    sf = move_client.send_goal_async(goal)
    rclpy.spin_until_future_complete(node, sf)
    handle = sf.result()
    if handle is None or not handle.accepted:
        return MoveItErrorCodes.PLANNING_FAILED, None
    rf = handle.get_result_async()
    rclpy.spin_until_future_complete(node, rf)
    res = rf.result().result
    return res.error_code.val, res.planned_trajectory


def plan_to_joint_goal(joint_values: dict, vel: float = 0.3, acc: float = 0.3,
                       plan_time: float = 10.0):
    req = make_plan_request(vel, acc, plan_time=plan_time)
    req.goal_constraints.append(make_joint_constraints(joint_values))
    code_val, traj = send_move_goal(req, plan_only=True)
    return code_val == MoveItErrorCodes.SUCCESS, traj


def plan_to_pose_goal(pose, vel: float = 0.3, acc: float = 0.3,
                      plan_time: float = 10.0):
    req = make_plan_request(vel, acc, plan_time=plan_time)
    c = Constraints()
    c.position_constraints.append(make_position_constraint(pose))
    c.orientation_constraints.append(make_orientation_constraint(pose))
    req.goal_constraints.append(c)
    code_val, traj = send_move_goal(req, plan_only=True)
    return code_val == MoveItErrorCodes.SUCCESS, traj

### 2-3. `trajectory_to_ee_path()` — FK 로 EE 경로 환산

`MoveGroup` 의 PRM / RRT 류 플래너가 만든 trajectory 는 조인트 공간에서의 점들이라
끝단이 실제로 어디로 끌려갈지 보려면 각 시점에 **순기구학 (FK)** 을 풀어야 한다.

`compute_fk` 서비스는 `(joint state, link name)` → `pose` 로 변환해 준다.
trajectory 의 모든 점을 풀면 비싸므로 `max_points` 로 다운샘플.

이게 **ex05 에는 없던 새 단계** — cartesian 직선 보간은 직선이 보장돼서 waypoints
만 그리면 됐지만, 일반 plan 은 곡선이라 trajectory 자체를 풀어 봐야 한다.

In [4]:
def trajectory_to_ee_path(trajectory, max_points: int = 60):
    '''RobotTrajectory → fr3_hand_tcp 의 base 기준 (x,y,z) 리스트.'''
    jt = trajectory.joint_trajectory
    total = len(jt.points)
    if total == 0:
        return []
    step = max(1, total // max_points)
    indices = list(range(0, total, step))
    if indices[-1] != total - 1:
        indices.append(total - 1)
    pts = []
    for idx in indices:
        req = GetPositionFK.Request()
        req.header.frame_id = REFERENCE_FRAME
        req.fk_link_names = [END_EFFECTOR_LINK]
        rs = RobotState()
        rs.joint_state.name = list(jt.joint_names)
        rs.joint_state.position = list(jt.points[idx].positions)
        req.robot_state = rs
        fut = fk_client.call_async(req)
        rclpy.spin_until_future_complete(node, fut)
        resp = fut.result()
        if resp and resp.error_code.val == MoveItErrorCodes.SUCCESS and resp.pose_stamped:
            p = resp.pose_stamped[0].pose.position
            pts.append((p.x, p.y, p.z))
    return pts

### 2-4. `publish_ee_path()` — RViz 미리보기 발행

`trajectory_to_ee_path` 의 결과를 `LINE_STRIP` 한 개로 발행한다.
ns / id 는 고정이라 다음 단계에서 새 경로를 발행하면 자동으로 덮어써진다 —
즉 매 단계마다 RViz 에는 **현재 실행할 경로 한 개** 만 보인다.

In [5]:
def publish_ee_path(ee_points, color=None):
    if not ee_points:
        return
    if color is None:
        color = COLOR_EE_PATH
    stamp = node.get_clock().now().to_msg()
    line = Marker()
    line.header.frame_id = REFERENCE_FRAME
    line.header.stamp = stamp
    line.ns = 'ee_path'
    line.id = 0
    line.type = Marker.LINE_STRIP
    line.action = Marker.ADD
    line.pose.orientation.w = 1.0
    line.scale.x = 0.008
    line.color = color
    line.points = [Point(x=p[0], y=p[1], z=p[2]) for p in ee_points]
    _markers.markers = [m for m in _markers.markers
                        if (m.ns, m.id) != ('ee_path', 0)]
    _markers.markers.append(line)
    marker_pub.publish(_markers)

### 2-5. `execute_trajectory()` — 계획된 trajectory 실행

ex05 와 동일. 계획과 실행이 분리되어 있어 사이에 미리보기 단계가 들어갈 수 있다.

In [6]:
def execute_trajectory(trajectory) -> bool:
    g = ExecuteTrajectory.Goal()
    g.trajectory = trajectory
    sf = execute_client.send_goal_async(g)
    rclpy.spin_until_future_complete(node, sf)
    handle = sf.result()
    if handle is None or not handle.accepted:
        return False
    rf = handle.get_result_async()
    rclpy.spin_until_future_complete(node, rf)
    return rf.result().result.error_code.val == MoveItErrorCodes.SUCCESS

### 2-6. `plan_viz_execute()` — 셋을 한 줄로 묶기

ex07 의 모든 모션이 이 함수 한 번 호출로 진행된다 —
**계획만** → **EE 경로 RViz 발행** → **실행**. 9 단계 시퀀스가 깔끔하게 펼쳐지는 비결.

In [7]:
def plan_viz_execute(pose, vel: float = 0.3, label: str = '') -> bool:
    ok, traj = plan_to_pose_goal(pose, vel=vel, acc=vel, plan_time=10.0)
    if not ok or traj is None:
        node.get_logger().error(f'{label}: 계획 실패')
        return False
    pts = trajectory_to_ee_path(traj)
    if pts:
        publish_ee_path(pts)
    return execute_trajectory(traj)


def plan_viz_execute_joint(joint_values: dict, vel: float = 0.3,
                           label: str = '') -> bool:
    ok, traj = plan_to_joint_goal(joint_values, vel=vel, acc=vel)
    if not ok or traj is None:
        node.get_logger().error(f'{label}: 계획 실패')
        return False
    pts = trajectory_to_ee_path(traj)
    if pts:
        publish_ee_path(pts)
    return execute_trajectory(traj)

## 3. 핵심을 쓰기 위한 설정

위 핵심 함수들이 참조하는 객체와 보조 헬퍼.

- ROS 2 초기화 / 노드 / 액션·서비스 클라이언트 / `joint_states` 구독 / 마커 퍼블리셔 / FK 클라이언트
- 서버·서비스·`/joint_states` 준비 대기
- SRDF 에서 `ready` 자세 읽기
- Pose 헬퍼 (Euler ↔ Quaternion, `make_pose`)
- MoveGroup 빌딩블록 — Constraints / `MotionPlanRequest`
- 그리퍼 헬퍼 — `gripper_open` / `gripper_close`
- RViz Pick / Place 테이블 마커 + EE 경로 색상 상수

### 3-1. ROS 2 초기화 + 노드 + 클라이언트

In [8]:
from rclpy.node import Node
from rclpy.action import ActionClient
from rclpy.parameter import Parameter
from sensor_msgs.msg import JointState
from visualization_msgs.msg import MarkerArray
from control_msgs.action import FollowJointTrajectory

try:
    rclpy.init()
except RuntimeError:
    pass

node = Node(
    'franka_ex07_pick_place_demo',
    parameter_overrides=[Parameter('use_sim_time', value=True)],
)
move_client    = ActionClient(node, MoveGroup, 'move_action')
execute_client = ActionClient(node, ExecuteTrajectory, 'execute_trajectory')
gripper_client = ActionClient(node, FollowJointTrajectory, GRIPPER_ACTION)
fk_client      = node.create_client(GetPositionFK, 'compute_fk')
marker_pub     = node.create_publisher(MarkerArray, MARKER_TOPIC, 10)

joint_state = {'msg': None}
node.create_subscription(
    JointState, 'joint_states',
    lambda msg: joint_state.update(msg=msg), 10,
)
node.get_logger().info('=== franka_ex07 노트북 노드 생성 완료 ===')

[INFO] [1783245918.839469102] [franka_ex07_pick_place_demo]: === franka_ex07 노트북 노드 생성 완료 ===


True

### 3-2. 액션 / 서비스 / `/joint_states` 준비 대기

In [9]:
import time

def wait_for_ready(timeout_sec: float = 30.0) -> None:
    if not move_client.wait_for_server(timeout_sec=timeout_sec):
        raise RuntimeError('MoveGroup 액션 서버 연결 실패')
    if not execute_client.wait_for_server(timeout_sec=timeout_sec):
        raise RuntimeError('ExecuteTrajectory 액션 서버 연결 실패')
    if not gripper_client.wait_for_server(timeout_sec=timeout_sec):
        raise RuntimeError('Gripper 액션 서버 연결 실패')
    if not fk_client.wait_for_service(timeout_sec=timeout_sec):
        raise RuntimeError('compute_fk 서비스 연결 실패')
    start = time.time()
    while joint_state['msg'] is None:
        rclpy.spin_once(node, timeout_sec=0.1)
        if time.time() - start > timeout_sec:
            raise RuntimeError('joint_states 수신 실패')
    node.get_logger().info(
        'move/execute/gripper action + compute_fk svc + /joint_states 준비됨'
    )

wait_for_ready()

[INFO] [1783245926.570200297] [franka_ex07_pick_place_demo]: move/execute/gripper action + compute_fk svc + /joint_states 준비됨


### 3-3. SRDF 에서 `ready` 자세 읽어오기

FR3 SRDF 에는 `home` 이 없고 `ready` / `extended` 만 있다.

In [10]:
from rclpy.parameter_client import AsyncParameterClient
import xml.etree.ElementTree as ET

def fetch_srdf_xml(timeout_sec: float = 10.0) -> str:
    client = AsyncParameterClient(node, 'move_group')
    if not client.wait_for_services(timeout_sec=timeout_sec):
        raise RuntimeError('move_group 파라미터 서비스 연결 실패')
    future = client.get_parameters(['robot_description_semantic'])
    rclpy.spin_until_future_complete(node, future, timeout_sec=timeout_sec)
    return future.result().values[0].string_value

def parse_named_pose(srdf_xml: str, name: str, group: str) -> dict:
    root = ET.fromstring(srdf_xml)
    for gs in root.findall('group_state'):
        if gs.attrib.get('group') == group and gs.attrib.get('name') == name:
            return {j.attrib['name']: float(j.attrib.get('value', '0'))
                    for j in gs.findall('joint')}
    raise RuntimeError(f'SRDF group_state "{name}" (group={group}) 없음')

def load_named_pose(name: str, timeout_sec: float = 10.0) -> dict:
    return parse_named_pose(fetch_srdf_xml(timeout_sec), name, PLANNING_GROUP)

ready_target = load_named_pose('ready')
node.get_logger().info(f'ready: {ready_target}')

[INFO] [1783245927.742529467] [franka_ex07_pick_place_demo]: ready: {'fr3_joint1': 0.0, 'fr3_joint2': -0.7853981633974483, 'fr3_joint3': 0.0, 'fr3_joint4': -2.356194490192345, 'fr3_joint5': 0.0, 'fr3_joint6': 1.5707963267948966, 'fr3_joint7': 0.7853981633974483}


True

### 3-4. Pose 헬퍼 — Euler ↔ Quaternion

In [11]:
import math
import tf_transformations
from geometry_msgs.msg import Pose, Quaternion

def euler_to_quaternion(roll: float, pitch: float, yaw: float) -> Quaternion:
    q = tf_transformations.quaternion_from_euler(roll, pitch, yaw)
    return Quaternion(x=q[0], y=q[1], z=q[2], w=q[3])

def make_pose(x: float, y: float, z: float,
              roll: float = 0.0, pitch: float = 0.0, yaw: float = 0.0) -> Pose:
    pose = Pose()
    pose.position = Point(x=x, y=y, z=z)
    pose.orientation = euler_to_quaternion(roll, pitch, yaw)
    return pose

### 3-5. MoveGroup 빌딩블록 — Constraints / `MotionPlanRequest`

`plan_to_*_goal()` 이 내부에서 호출하는 헬퍼들.

In [12]:
from moveit_msgs.msg import (
    JointConstraint,
    PositionConstraint, OrientationConstraint, BoundingVolume,
)
from shape_msgs.msg import SolidPrimitive
from geometry_msgs.msg import Vector3

def make_joint_constraints(joint_values: dict, tol: float = 0.01) -> Constraints:
    c = Constraints()
    for jname, val in joint_values.items():
        c.joint_constraints.append(JointConstraint(
            joint_name=jname, position=val,
            tolerance_above=tol, tolerance_below=tol, weight=1.0,
        ))
    return c

def make_position_constraint(pose: Pose, tol: float = 0.01) -> PositionConstraint:
    pc = PositionConstraint()
    pc.header.frame_id = REFERENCE_FRAME
    pc.link_name = END_EFFECTOR_LINK
    pc.target_point_offset = Vector3(x=0.0, y=0.0, z=0.0)
    bv = BoundingVolume()
    sphere = SolidPrimitive()
    sphere.type = SolidPrimitive.SPHERE
    sphere.dimensions = [tol]
    bv.primitives.append(sphere)
    sp = Pose()
    sp.position = Point(x=pose.position.x, y=pose.position.y, z=pose.position.z)
    sp.orientation.w = 1.0
    bv.primitive_poses.append(sp)
    pc.constraint_region = bv
    pc.weight = 1.0
    return pc

def make_orientation_constraint(pose_or_quat, tol: float = 0.01) -> OrientationConstraint:
    oc = OrientationConstraint()
    oc.header.frame_id = REFERENCE_FRAME
    oc.link_name = END_EFFECTOR_LINK
    if hasattr(pose_or_quat, 'orientation'):
        oc.orientation = pose_or_quat.orientation
    else:
        oc.orientation = pose_or_quat
    oc.absolute_x_axis_tolerance = tol
    oc.absolute_y_axis_tolerance = tol
    oc.absolute_z_axis_tolerance = tol
    oc.weight = 1.0
    return oc

def make_plan_request(vel: float = 0.3, acc: float = 0.3,
                      attempts: int = 5, plan_time: float = 10.0,
                      planner_id: str = '') -> MotionPlanRequest:
    req = MotionPlanRequest()
    req.group_name = PLANNING_GROUP
    req.num_planning_attempts = attempts
    req.allowed_planning_time = plan_time
    req.max_velocity_scaling_factor = vel
    req.max_acceleration_scaling_factor = acc
    if planner_id:
        req.planner_id = planner_id
    return req

### 3-6. 그리퍼 헬퍼

ex06 와 동일한 `FollowJointTrajectory` 패턴. `fr3_finger_joint1` 단독 명령
(`fr3_finger_joint2` 는 URDF `<mimic>` 으로 자동 추종).

In [13]:
from trajectory_msgs.msg import JointTrajectory, JointTrajectoryPoint
from builtin_interfaces.msg import Duration

def move_gripper(position: float, duration_sec: float = 1.0) -> bool:
    '''fr3_finger_joint1 을 position 으로 이동 (0.0=닫힘, 0.04=열림).'''
    pos = float(max(0.0, min(GRIPPER_OPEN, position)))
    g = FollowJointTrajectory.Goal()
    g.trajectory = JointTrajectory()
    g.trajectory.joint_names = [GRIPPER_JOINT]
    pt = JointTrajectoryPoint()
    pt.positions = [pos]
    pt.time_from_start = Duration(
        sec=int(duration_sec),
        nanosec=int((duration_sec - int(duration_sec)) * 1e9),
    )
    g.trajectory.points.append(pt)
    sf = gripper_client.send_goal_async(g)
    rclpy.spin_until_future_complete(node, sf)
    handle = sf.result()
    if handle is None or not handle.accepted:
        node.get_logger().error('gripper goal 거부')
        return False
    rf = handle.get_result_async()
    rclpy.spin_until_future_complete(node, rf)
    code_val = rf.result().result.error_code
    ok = (code_val == 0)
    node.get_logger().info(f'gripper {pos*1000:.1f}mm 이동 (code={code_val})')
    return ok

def gripper_open():  return move_gripper(GRIPPER_OPEN)
def gripper_close(): return move_gripper(GRIPPER_CLOSED)

### 3-7. RViz 마커 — Pick / Place 테이블 + EE 경로 색상

| 색상 | 의미 |
|---|---|
| 초록 (`COLOR_PICK`) | Pick 위치 큐브 |
| 파랑 (`COLOR_PLACE`) | Place 위치 큐브 |
| 흰색 (`COLOR_TEXT`) | 라벨 텍스트 |
| 주황 (`COLOR_EE_PATH`) | 실행 직전 EE 경로 미리보기 (핵심에서 참조) |

In [14]:
from std_msgs.msg import ColorRGBA

COLOR_PICK    = ColorRGBA(r=0.3, g=0.7, b=0.3, a=0.6)
COLOR_PLACE   = ColorRGBA(r=0.3, g=0.3, b=0.8, a=0.6)
COLOR_TEXT    = ColorRGBA(r=1.0, g=1.0, b=1.0, a=1.0)
COLOR_EE_PATH = ColorRGBA(r=1.0, g=0.5, b=0.0, a=0.95)

_markers = MarkerArray()

def add_table_marker(pos, label: str, idx: int, color: ColorRGBA):
    stamp = node.get_clock().now().to_msg()
    cube = Marker()
    cube.header.frame_id = REFERENCE_FRAME
    cube.header.stamp = stamp
    cube.ns = 'tables'
    cube.id = idx
    cube.type = Marker.CUBE
    cube.action = Marker.ADD
    cube.pose.position = Point(x=pos[0], y=pos[1], z=pos[2] - 0.06)
    cube.pose.orientation.w = 1.0
    cube.scale = Vector3(x=0.12, y=0.12, z=0.10)
    cube.color = color
    text = Marker()
    text.header.frame_id = REFERENCE_FRAME
    text.header.stamp = stamp
    text.ns = 'labels'
    text.id = idx
    text.type = Marker.TEXT_VIEW_FACING
    text.action = Marker.ADD
    text.pose.position = Point(x=pos[0], y=pos[1], z=pos[2] + 0.10)
    text.pose.orientation.w = 1.0
    text.scale.z = 0.05
    text.color = COLOR_TEXT
    text.text = label
    keys = {('tables', idx), ('labels', idx)}
    _markers.markers = [m for m in _markers.markers if (m.ns, m.id) not in keys]
    _markers.markers.extend([cube, text])
    marker_pub.publish(_markers)

## 4. Pick & Place 시나리오 (9 단계)

여기부터는 위에서 정의한 `plan_viz_execute()` / `plan_viz_execute_joint()` 와
`gripper_*` 만 호출한다. 각 단계 셀은 한두 줄.

**Approach / Lift / Retreat 패턴**: 직접 Pick/Place 위치로 가지 않고, 같은 X-Y 위 ~15 cm
높이로 먼저 이동(접근) → 하강 → 잡기 → 다시 위로 들어올림 → 운반 → 하강 → 놓기 → 후퇴.
충돌 위험을 줄이는 산업용 표준 모션.

### 4-1. Pick / Place 위치 + RViz 표시

In [15]:
pick_pos  = (0.50, 0.00, 0.30)   # base 기준 ~50 cm 전방
place_pos = (0.00, 0.50, 0.30)   # base 기준 ~50 cm 좌측
APPROACH_HEIGHT = 0.45            # 접근 / 운반 z

add_table_marker(pick_pos,  'Pick',  0, COLOR_PICK)
add_table_marker(place_pos, 'Place', 1, COLOR_PLACE)
node.get_logger().info('Pick / Place 마커 발행')

[INFO] [1783245936.803199675] [franka_ex07_pick_place_demo]: Pick / Place 마커 발행


True

### 4-2. 1단계 — `ready` 자세 + 그리퍼 열기

In [16]:
node.get_logger().info('--- 1단계: ready 자세 + 그리퍼 열기 ---')
plan_viz_execute_joint(ready_target, vel=0.3, label='Ready')
gripper_open()
time.sleep(1.0)

[INFO] [1783245942.301537051] [franka_ex07_pick_place_demo]: --- 1단계: ready 자세 + 그리퍼 열기 ---
[INFO] [1783245943.533001887] [franka_ex07_pick_place_demo]: gripper 40.0mm 이동 (code=0)


### 4-3. 2단계 — Pick 위치 위로 접근

같은 X-Y 위에서 ~15cm 위로 먼저 간다. 마지막 하강을 직선화해 충돌 위험을 줄인다.

In [17]:
approach_pick = make_pose(pick_pos[0], pick_pos[1], APPROACH_HEIGHT,
                          math.pi, 0.0, 0.0)
node.get_logger().info('--- 2단계: Pick 접근 자세 ---')
plan_viz_execute(approach_pick, label='ApproachPick')
time.sleep(0.5)

[INFO] [1783245945.505506856] [franka_ex07_pick_place_demo]: --- 2단계: Pick 접근 자세 ---


### 4-4. 3단계 — Pick 위치로 하강

In [18]:
pick_pose = make_pose(*pick_pos, math.pi, 0.0, 0.0)
node.get_logger().info('--- 3단계: Pick 위치 하강 ---')
plan_viz_execute(pick_pose, vel=0.2, label='Pick')
time.sleep(0.5)

[INFO] [1783245948.400146301] [franka_ex07_pick_place_demo]: --- 3단계: Pick 위치 하강 ---


### 4-5. 4단계 — 그리퍼 닫기 (가상 물체 잡기)

In [19]:
node.get_logger().info('--- 4단계: 그리퍼 닫기 ---')
gripper_close()
time.sleep(1.0)

[INFO] [1783245950.998085614] [franka_ex07_pick_place_demo]: --- 4단계: 그리퍼 닫기 ---
[INFO] [1783245952.001377647] [franka_ex07_pick_place_demo]: gripper 0.0mm 이동 (code=0)


### 4-6. 5단계 — 들어올리기

In [20]:
lift = make_pose(pick_pos[0], pick_pos[1], APPROACH_HEIGHT,
                 math.pi, 0.0, 0.0)
node.get_logger().info('--- 5단계: 물체 들어올리기 ---')
plan_viz_execute(lift, vel=0.2, label='Lift')
time.sleep(0.5)

[INFO] [1783245953.226490534] [franka_ex07_pick_place_demo]: --- 5단계: 물체 들어올리기 ---


### 4-7. 6단계 — Place 위치 위로 운반

운반 중에는 yaw 를 90° 돌려 손목 방향이 자연스럽게 따라가도록 한다 — 7-DOF redundancy 활용.

In [21]:
approach_place = make_pose(place_pos[0], place_pos[1], APPROACH_HEIGHT,
                           math.pi, 0.0, math.pi / 2)
node.get_logger().info('--- 6단계: Place 위치 위로 운반 ---')
plan_viz_execute(approach_place, label='Transport')
time.sleep(0.5)

[INFO] [1783245955.711939917] [franka_ex07_pick_place_demo]: --- 6단계: Place 위치 위로 운반 ---


### 4-8. 7단계 — Place 위치로 하강

In [22]:
place_pose = make_pose(*place_pos, math.pi, 0.0, math.pi / 2)
node.get_logger().info('--- 7단계: Place 하강 ---')
plan_viz_execute(place_pose, vel=0.2, label='Place')
time.sleep(0.5)

[INFO] [1783245966.078931053] [franka_ex07_pick_place_demo]: --- 7단계: Place 하강 ---


### 4-9. 8단계 — 그리퍼 열기 (놓기) + 후퇴

In [23]:
node.get_logger().info('--- 8단계: 그리퍼 열기 (놓기) ---')
gripper_open()
time.sleep(1.0)

retreat = make_pose(place_pos[0], place_pos[1], APPROACH_HEIGHT,
                    math.pi, 0.0, math.pi / 2)
node.get_logger().info('--- 후퇴 ---')
plan_viz_execute(retreat, label='Retreat')
time.sleep(0.5)

[INFO] [1783245968.121561854] [franka_ex07_pick_place_demo]: --- 8단계: 그리퍼 열기 (놓기) ---
[INFO] [1783245969.124620460] [franka_ex07_pick_place_demo]: gripper 40.0mm 이동 (code=0)
[INFO] [1783245970.126841778] [franka_ex07_pick_place_demo]: --- 후퇴 ---


### 4-10. 9단계 — `ready` 복귀

In [24]:
node.get_logger().info('--- 9단계: ready 복귀 ---')
plan_viz_execute_joint(ready_target, vel=0.3, label='Ready')
node.get_logger().info('=== franka_ex07 완료! ===')

[INFO] [1783245971.805808310] [franka_ex07_pick_place_demo]: --- 9단계: ready 복귀 ---
[INFO] [1783245974.737196200] [franka_ex07_pick_place_demo]: === franka_ex07 완료! ===


True

## 5. 정리

In [25]:
node.destroy_node()
try:
    rclpy.shutdown()
except Exception:
    pass